In [1]:
# ── Cell 0: Setup (chạy đầu tiên) ──
import os

# Clone repo nếu chưa có
if not os.path.exists('/content/project'):
    !git clone https://github.com/TruongDuyLongPTIT/CTQW_PRO_METABOLITES_PRIORITIZING.git /content/project

# Add src vào Python path
import sys
sys.path.insert(0, '/content/project/src')

# Install dependencies nếu cần
!pip install -q torch scikit-learn networkx tqdm

Cloning into '/content/project'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 98 (delta 51), reused 41 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 994.88 KiB | 9.13 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print("Drive đã mount sẵn.")

Mounted at /content/drive


In [3]:
!python /content/project/experiments/01_main_results.py

STEP 1 — Build graph
  Graph: 2788 nodes, 22439 edges
  G_pro: 2894 nodes (106 pathway), 31360 edges

STEP 2 — Build eval sets
  hmdb_to_recon: +0 IK, +331 name → 3286 total
  Extracting SMPDB metabolites...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:39<00:00, 1225.29it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  eval_set1 (HMDB+CTD): 158 diseases
  eval_set2 (MarkerDB): 21 diseases
  eval_set3 (SMPDB):    153 diseases

STEP 3 — Eigendecomposition
  Done.

  Device: cpu

[Table 1] RWR vs CTQW on G_cc...
RWR/HMDB+CTD: 100% 158/158 [11:33<00:00,  4.39s/it]
CTQW/HMDB+CTD: 100% 158/158 [10:34<00:00,  4.02s/it]
  HMDB+CTD: 22.1 min
RWR/MarkerDB: 100% 21/21 [02:04<00:00,  5.95s/it]
CTQW/MarkerDB: 100% 21/21 [01:54<00:00,  5.47s/it]
  MarkerDB: 4.0 min
RWR/SMPDB: 100% 153/153 [08:48<00:00,  3.46s/it]
CTQW/SMPDB: 100% 153/153 [07:57<00:00,  3.12s/it]
  SMPDB: 16.8 min

[Table 2] PROFANCY vs CTQW-PRO on G_pro...
PROFANCY/HMDB+CTD: 100% 158/158 [10:

In [4]:
!python /content/project/experiments/02_ablation_graph.py

Building graphs...
  G_pro: 2894 → clean: 2851 nodes
Building eval sets...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:43<00:00, 1109.57it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
Eigendecomposition...

Running ablation...
PROF_o/HMDB+CTD: 100% 158/158 [10:47<00:00,  4.10s/it]
CTQW_o/HMDB+CTD: 100% 158/158 [10:43<00:00,  4.08s/it]
PROF_c/HMDB+CTD: 100% 158/158 [15:54<00:00,  6.04s/it]
CTQW_c/HMDB+CTD: 100% 158/158 [11:33<00:00,  4.39s/it]
  HMDB+CTD: 49.0 min
PROF_o/MarkerDB: 100% 21/21 [02:01<00:00,  5.78s/it]
CTQW_o/MarkerDB: 100% 21/21 [02:11<00:00,  6.26s/it]
PROF_c/MarkerDB: 100% 21/21 [02:50<00:00,  8.12s/it]
CTQW_c/MarkerDB: 100% 21/21 [02:06<00:00,  6.04s/it]
  MarkerDB: 9.2 min
PROF_o/SMPDB: 100% 153/153 [08:06<00:00,  3.18s/it]
CTQW_o/SMPDB: 100% 153/153 [08:56<00:00,  3.51s/it]
PROF_c/SMPDB: 100% 153/153 [12:05<00:00,  4.74s/it]
CTQW_c/SMPDB: 100% 153/153 [08:41<00:00,  3.41s/it]
  SMPDB: 37.8 min

ABLATION: Original vs Clean G_

In [5]:
!python /content/project/experiments/03_negative_results.py

Setup...
  Disease pathways: 20248
SMPDB files: 100% 48687/48687 [00:42<00:00, 1150.71it/s]
  Loaded 20244 pathways (errors=0)
  Dedup: 20244 → 3240 → 362 → 215
  HMDB+CTD: 158 diseases
  SMPDB:    153 diseases
  Device: cpu

EXP 1: Self-loop leakage analysis
  (A) Baseline:     CTQW-PRO, γ=0, no self-loop
  (B) Leaked:       eigh(A_pro + γ·diag(ALL mets)) — test_met in diagonal
  (C) Leakage-free: eigh(A_pro + γ·diag(seeds_only)) — per LOO fold

  Sample: 10 diseases, γ=10.0

  Condition                           AUC      MRR     R@20
  ----------------------------------------------------------
  (A) Baseline (no self-loop)      0.9515   0.2931   0.6203
  (B) Self-loop LEAKED             0.9607   0.3927   0.6208  ← inflated
  (C) Self-loop leakage-free       0.9503   0.2918   0.5812

  ΔMRR (B vs A): +0.0996  ← gap do leakage
  ΔMRR (C vs A): -0.0012  ← self-loop thực sự
  → Leakage artifact confirmed: B inflate hơn C bởi 0.1008 MRR
  NOTE: Sample 10 diseases — kết luận cuối cần chạy 